In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import openpyxl

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

In [2]:
#runners = pd.read_excel("../data/cleaned/results/runners.xlsx")
races = pd.read_excel("../../data/cleaned/results/races.xlsx")

In [3]:
runners = pd.read_csv("../../data/cleaned/results/runners.csv")

C:\Users\sriva\AppData\Local\Temp\ipykernel_19076\4033751346.py:1: DtypeWarning: Columns (0: sire_nat, 1: dam_nat, 2: weight) have mixed types. Specify dtype option on import or set low_memory=False.
  runners = pd.read_csv("../../data/cleaned/results/runners.csv")


In [4]:
print("runners:", type(runners))
print("races:  ", type(races))

print("\nRUNNERS")
print(runners.shape)
print(runners.columns.tolist())

print("\nRACES")
print(races.shape)
print(races.columns.tolist())

runners: <class 'pandas.DataFrame'>
races:   <class 'pandas.DataFrame'>

RUNNERS
(68201, 17)
['meet_date', 'venue', 'race_no', 'placing', 'horse_name', 'horse_seq', 'sire', 'sire_nat', 'dam', 'dam_nat', 'weight', 'jockey', 'jockey_claim', 'trainer', 'odds', 'finish_time', 'horse_body_wt']

RACES
(7137, 21)
['meet_date', 'venue', 'race_no', 'card_seq', 'race_name', 'class_conditions', 'scheduled_time', 'distance_text', 'distance_meters', 'video_url', 'ownership', 'breeder', 'margins', 'results_by_card', 'tote_favourite', 'win_div', 'place_div', 'shp_div', 'for_div', 'qnl_div', 'tnl_div']


In [5]:
display(runners.head(20))

print("\nDtypes:")
display(runners.dtypes)

print("\nNull percentage:")
display(
    runners.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .round(2)
    .to_frame("null_pct")
)

,meet_date,venue,race_no,placing,horse_name,horse_seq,sire,sire_nat,dam,dam_nat,weight,jockey,jockey_claim,trainer,odds,finish_time,horse_body_wt
0,15-07-2010,Mumbai,1,1,RISING GLORY,12346,Black Cash,NaN,Serious Trouble,NaN,57,Imran Chisty,NaN,H. J. Antia,11-Apr,1:01:033,429
1,15-07-2010,Mumbai,1,2,CANDY FLOSS,12432,Brave Hunter,NaN,Cagliari,NaN,55.5,P. Trevor,3.5,S. K. Sunderji,07-Jan,1:01:050,448
2,15-07-2010,Mumbai,1,3,CHINGARI,12480,Ontario,NaN,Miss Formidare,NaN,55.5,Ladjadj Stephane,NaN,Narendra Lagad,13-Feb,1:02:002,413
3,15-07-2010,Mumbai,1,4,BENEDICTUS,12199,Puerto Madero,NaN,Innara,NaN,57,S. S. Baria,NaN,Altaf Hussain,12-Jan,1:02:010,452
4,15-07-2010,Mumbai,1,5,THE FIRST LADY,11701,Diffident,NaN,Ampula,NaN,58.5,C. Rajendra,NaN,Faisal A. Abbas,05-Jan,1:02:011,416
5,15-07-2010,Mumbai,1,6,ASHWA PRAHAR,11735,Serious Spender,NaN,Sweet Memento,NaN,55.5,Dashrath Singh,NaN,Sangramsinh N. Joshi,12-Jan,1:02:035,444
6,15-07-2010,Mumbai,1,7,DANCING MONEY,12360,Mr. Mellon,NaN,Shirley Valentine,NaN,48.5,N. S. Parmar,2.5,D. J. Surti,11-Feb,1:02:084,390
7,15-07-2010,Mumbai,1,8,MGDIAN,12314,Case Law,NaN,Jaipur Jewel,NaN,58.5,C. S. Jodha,NaN,Magansingh P. Jodha,20-Jan,1:02:093,482
8,15-07-2010,Mumbai,2,1,BLAU HIMMEL,11601,Royal Kingdom,NaN,Calais,NaN,61,M. Narredu,NaN,Dallas Todywalla,07-Jan,1:28:059,413
9,15-07-2010,Mumbai,2,2,ROSES ALL THE WAY,11587,Serious Spender,NaN,Perfect Storm,NaN,49.5,Dashrath Singh,NaN,Sangramsinh N. Joshi,06-Jan,1:28:063,452



Dtypes:


meet_date            str
venue                str
race_no            int64
placing              str
horse_name           str
horse_seq          int64
sire                 str
sire_nat             str
dam                  str
dam_nat              str
weight            object
jockey               str
jockey_claim     float64
trainer              str
odds                 str
finish_time          str
horse_body_wt        str
dtype: object


Null percentage:


,null_pct
dam_nat,95.28
sire_nat,89.87
jockey_claim,78.41
jockey,0.27
meet_date,0.00
horse_name,0.00
venue,0.00
placing,0.00
race_no,0.00
dam,0.00


In [6]:
display(races.head(20))

print("\nDtypes:")
display(races.dtypes)

print("\nNull percentage:")
display(
    races.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .round(2)
    .to_frame("null_pct")
)

,meet_date,venue,race_no,card_seq,race_name,class_conditions,scheduled_time,distance_text,distance_meters,video_url,ownership,breeder,margins,results_by_card,tote_favourite,win_div,place_div,shp_div,for_div,qnl_div,tnl_div
0,2010-07-15,Mumbai,1,1,The Everything Plate,"Class V; H'cap, Indian Horses rated 1 to 19.",2.00 P.M.,(About) 1000 Metres.,1000,http://www.mumbairaces.com/index.php?chan=2&cat=1,Mr H. J. Antia's,Nanoli Stud & Agricultural Farms,"1 , 3 1/4, 1/2, Shd, 1 1/2, 3 , 1/2,",5-1-7-4-3-6-8-2,DANCING MONEY,34,"13,17,20",52,252,94,647 & 359
1,2010-07-15,Mumbai,2,2,The Princess Gombos Plate,"Class V; H'cap, Indian Horses 5 years old and over, rated 1 to 26.",2.30 P.M.,(About) 1400 Metres.,1400,http://www.mumbairaces.com/index.php?chan=2&cat=1,Mr & Mrs N. D. Talati's,Five Stars Shipping Company Private Limited,"nk, 3 3/4, 1/2, 1 1/2, 3/4, 5 3/4,",1-7-2-4-6-3-5,HI DOLLY,49,"25,22",34,129,39,800 & 487
2,2010-07-15,Mumbai,3,3,The Storm Plate - Division II,"Terms, Maiden Indian Horses, 3 years old only.",3.00 P.M.,(About) 1200 Metres.,1200,http://www.mumbairaces.com/index.php?chan=2&cat=1,"M/s Rajeev D. Raut, S. T. Shivaprasad, Banwait J. Singh & Southern P'dise St...",Southern Paradise Stud & Developers Farms Pvt Ltd,"2 , nk, 6 1/4, 3 3/4, 4 1/4, 3 1/2, nk, 3/4,",9-6-5-7-8-2-1-4-3,SHOWSTOPPER,15,"11,15,12",52,54,63,84 & 22
3,2010-07-15,Mumbai,4,4,The Cupid Plate,"Class II; H'cap, Horses 5 years old and over, rated 60 to 86.",3.30 P.M.,(About) 1000 Metres.,1000,http://www.mumbairaces.com/index.php?chan=2&cat=1,Mr & Mrs Shapoor P. Mistry and Mr & Mrs Cyrus P. Mistry rep. Manjri Horse Br...,Manjri Stud Farm Pvt Ltd,"1/2, 2 1/4, nk, 1/2, 1/2, 3 , 2 1/2, 10 , 4 1/4,",1-9-2-3-5-10-8-6-7-4,HIGHLAND FLAME,35,"13,13,13",43,104,40,165 & 56
4,2010-07-15,Mumbai,5,5,The Storm Plate - Division I,"Terms, Maiden Indian Horses, 3 years old only.",4.00 P.M.,(About) 1200 Metres.,1200,http://www.mumbairaces.com/index.php?chan=2&cat=1,Mr & Mrs Shapoor P. Mistry rep. Manjri Horse Breeders' Farm Pvt Ltd & Mr Sha...,"Khan Mr Shahrooq, Khan Mrs Shahrooq","3/4, 3/4, 5 3/4, 2 , 1 1/4, 1 1/2, nk, 11 ,",8-2-6-4-1-7-5-3-9,OCEAN STAR,27,"11,16,10",42,221,122,137 & 35
5,2010-07-15,Mumbai,6,6,The Vibrant Plate - Division II,"Class IV; H'cap, Indian Horses rated 20 to 46.",4.30 P.M.,(About) 1000 Metres.,1000,http://www.mumbairaces.com/index.php?chan=2&cat=1,"Mr Nandlal Khandelwal, Kr Digvijay Singh Shekhawat & Mr Magansingh P. Jodha's",Jai-Govind Stud & Agricultural Farm,"Hd, 1 , 1 , 1 1/2, Shd, nk, 1 1/4, 3/4, Hd, 1 , Shd, 1 ,",2-11-7-1-9-8-3-4-13-12-6-5-10,MYPETHONEY,36,"12,26,14,18",85,249,170,850 & 289
6,2010-07-15,Mumbai,7,7,The Rose Royal Trophy,"Class I; H'cap, Horses rated 80 and upward.",5.00 P.M.,(About) 1200 Metres.,1200,http://www.mumbairaces.com/index.php?chan=2&cat=1,M/s Gautam Thapar & Sultan Singh rep. Sohna Stud Farm Pvt. Ltd's,Sohna Stud Farm Private Limited,"1 1/2, 1 3/4, nk, 3/4, Shd, 1/2, 2 3/4, Hd, 1/2, Shd, Shd,",5-11-9-1-8-12-3-2-10-6-4-7,ARTS,18,"12,12,13,48",39,37,28,142 & 74
7,2010-07-15,Mumbai,8,8,The Vibrant Plate - Division I,"Class IV; H'cap, Indian Horses rated 20 to 46.",5.30 P.M.,(About) 1000 Metres.,1000,http://www.mumbairaces.com/index.php?chan=2&cat=1,Ms.Christaline Glenn & Mr.Mehernosh Deboo rep. Aquarius Maritime Pvt Ltd's,Chanhill Farms,"2 1/4, 3 1/2, 1/2, Shd, 2 1/2, 1 1/2, nk, 2 , 2 1/4, 1 , Shd, 2 3/4,",1-8-7-2-13-6-10-4-9-12-3-11-5,LITTLE WARRIOR,23,"12,20,27,11",83,174,377,2385 & 17520
8,2010-07-16,Mumbai,9,1,The Wiggles Plate - Division II,"Class V; H'cap, Indian Horses rated 1 to 26.",1.00 P.M.,(About) 1200 Metres.,1200,http://www.mumbairaces.com/index.php?chan=2&cat=2,"Chetak Horse Racing Pvt Ltd, Begum Shaherbanoo Lagad & Mr Rajeev D. Raut's",Chanhill Farms,"1 1/2, 1 1/2, 2 3/4, 1 1/4, 2 1/4, 2 , 4 1/4,",5-1-7-6-3-8-4-2,POUND FOOLISH,71,"15,11,12",41,139,33,201 & 120
9,2010-07-16,Mumbai,10,2,The Wiggles Plate - Division I,"Class V; H'cap, Indian Horses rated 1 to 26.",1.30 P.M.,(About) 1200 Metres.,1200,http:/


Dtypes:


meet_date           datetime64[us]
venue                          str
race_no                      int64
card_seq                     int64
race_name                      str
class_conditions               str
scheduled_time                 str
distance_text                  str
distance_meters              int64
video_url                      str
ownership                      str
breeder                        str
margins                        str
results_by_card             object
tote_favourite                 str
win_div                     object
place_div                   object
shp_div                     object
for_div                     object
qnl_div                     object
tnl_div                     object
dtype: object


Null percentage:


,null_pct
qnl_div,2.96
tnl_div,2.89
tote_favourite,2.68
place_div,2.65
for_div,2.21
shp_div,1.58
win_div,1.35
video_url,0.29
ownership,0.11
breeder,0.10


In [7]:
print("RUNNERS")
print("Rows:", len(runners))

if "meet_date" in runners.columns:
    print("Dates:", runners["meet_date"].min(), "→", runners["meet_date"].max())

if "venue" in runners.columns:
    print("Venues:", runners["venue"].nunique())
    print(runners["venue"].value_counts().head(20))

print("\nRACES")
print("Rows:", len(races))

if "meet_date" in races.columns:
    print("Dates:", races["meet_date"].min(), "→", races["meet_date"].max())

if "venue" in races.columns:
    print("Venues:", races["venue"].nunique())
    print(races["venue"].value_counts().head(20))

RUNNERS
Rows: 68201
Dates: 01-01-2012 → 31-12-2023
Venues: 2
venue
Mumbai    43303
Pune      24898
Name: count, dtype: int64

RACES
Rows: 7137
Dates: 2010-07-15 00:00:00 → 2026-03-08 00:00:00
Venues: 2
venue
Mumbai    4556
Pune      2581
Name: count, dtype: int64


In [8]:
runners["meet_date_parsed"] = pd.to_datetime(
    runners["meet_date"],
    dayfirst=True,
    errors="coerce"
)

races["meet_date_parsed"] = pd.to_datetime(
    races["meet_date"],
    dayfirst=True,
    errors="coerce"
)

print("Runner invalid dates:", runners["meet_date_parsed"].isna().sum())
print("Race invalid dates:  ", races["meet_date_parsed"].isna().sum())

print(
    runners["meet_date_parsed"].min(),
    "→",
    runners["meet_date_parsed"].max()
)

print(
    races["meet_date_parsed"].min(),
    "→",
    races["meet_date_parsed"].max()
)

Runner invalid dates: 0
Race invalid dates:   0
2010-07-15 00:00:00 → 2026-03-08 00:00:00
2010-07-15 00:00:00 → 2026-03-08 00:00:00


In [9]:
race_key = ["meet_date_parsed", "venue", "race_no"]

race_groups = (
    runners
    .groupby(race_key, dropna=False)
    .size()
    .reset_index(name="runner_count")
)

print("Unique races represented:", len(race_groups))

print("\nRunner count distribution:")
display(race_groups["runner_count"].describe())

print("\nRaces with suspiciously small fields:")
display(
    race_groups[
        race_groups["runner_count"] < 3
    ].sort_values("runner_count")
)

print("\nRaces with large fields:")
display(
    race_groups[
        race_groups["runner_count"] > 20
    ].sort_values("runner_count", ascending=False).head(30)
)

Unique races represented: 7135

Runner count distribution:


count    7135.000000
mean        9.558655
std         3.226285
min         2.000000
25%         7.000000
50%         9.000000
75%        12.000000
max        22.000000
Name: runner_count, dtype: float64


Races with suspiciously small fields:


,meet_date_parsed,venue,race_no,runner_count
824,2011-10-15,Pune,212,2
872,2011-11-20,Mumbai,17,2
1006,2012-01-19,Mumbai,151,2
1394,2012-09-22,Pune,154,2
3328,2015-11-22,Mumbai,8,2
3591,2016-03-26,Mumbai,271,2
4049,2017-02-19,Mumbai,173,2
4517,2018-02-10,Mumbai,165,2
4546,2018-02-25,Mumbai,194,2



Races with large fields:


,meet_date_parsed,venue,race_no,runner_count
434,2011-02-06,Mumbai,184,22
437,2011-02-06,Mumbai,187,22
505,2011-03-06,Mumbai,255,22
1031,2012-02-04,Mumbai,176,22
2112,2013-10-26,Pune,246,22
1883,2013-07-19,Pune,17,22
3676,2016-07-28,Pune,5,22
3912,2016-12-04,Mumbai,36,22
2116,2013-10-26,Pune,250,22
2125,2013-10-27,Pune,259,22


In [10]:
runner_key = [
    "meet_date_parsed",
    "venue",
    "race_no",
    "horse_name"
]

duplicates = runners[
    runners.duplicated(
        subset=runner_key,
        keep=False
    )
].sort_values(runner_key)

print("Duplicate runner rows:", len(duplicates))

display(duplicates.head(50))

Duplicate runner rows: 0


,meet_date,venue,race_no,placing,horse_name,horse_seq,sire,sire_nat,dam,dam_nat,weight,jockey,jockey_claim,trainer,odds,finish_time,horse_body_wt,meet_date_parsed


In [11]:
print("Unique placing values:")
display(
    runners["placing"]
    .astype(str)
    .str.strip()
    .value_counts()
    .head(50)
)

Unique placing values:


placing
1       7144
2       7132
3       7115
4       7032
5       6701
6       6268
7       5736
8       5051
9       4152
10      3245
11      2362
12      1657
WD      1115
13      1079
14       705
DNC      372
15       369
WDRN     253
16       210
17       127
DNF       83
18        80
19        50
NDS       36
-         32
20        30
NPR       21
DQ        18
21        12
NS         9
22         5
Name: count, dtype: int64

In [12]:
placing_numeric = pd.to_numeric(
    runners["placing"],
    errors="coerce"
)

print("Numeric placings:", placing_numeric.notna().sum())
print("Non-numeric placings:", placing_numeric.isna().sum())

print("\nNon-numeric examples:")
display(
    runners.loc[
        placing_numeric.isna(),
        ["meet_date", "venue", "race_no", "horse_name", "placing"]
    ].head(50)
)

Numeric placings: 66262
Non-numeric placings: 1939

Non-numeric examples:


,meet_date,venue,race_no,horse_name,placing
125,16-07-2010,Mumbai,14,TSESEBE,-
149,16-07-2010,Mumbai,15,GLOWING STAR,WD
150,16-07-2010,Mumbai,15,STAR VISION,WD
160,16-07-2010,Mumbai,16,SECRET MAGIC,WD
171,16-07-2010,Mumbai,17,SEPIA TONE,WD
219,22-07-2010,Mumbai,22,SWEATY BETTY,WD
258,23-07-2010,Mumbai,27,STAR PRESENTATION,WD
313,01-08-2010,Mumbai,34,SWISS BELLE,WD
330,01-08-2010,Mumbai,37,RIVER KNIGHT,-
341,01-08-2010,Mumbai,37,TEQUILA,DQ


In [13]:
tmp = runners.copy()

tmp["placing_num"] = pd.to_numeric(
    tmp["placing"],
    errors="coerce"
)

winner_counts = (
    tmp.groupby(race_key)["placing_num"]
    .apply(lambda x: (x == 1).sum())
    .reset_index(name="winner_count")
)

print("Winner count distribution:")
display(winner_counts["winner_count"].value_counts().sort_index())

print("\nRaces with zero winners:")
display(
    winner_counts[winner_counts["winner_count"] == 0]
    .head(30)
)

print("\nRaces with multiple winners:")
display(
    winner_counts[winner_counts["winner_count"] > 1]
    .head(30)
)

Winner count distribution:


winner_count
0       4
1    7118
2      13
Name: count, dtype: int64


Races with zero winners:


,meet_date_parsed,venue,race_no,winner_count
1503,2012-11-22,Mumbai,10,0
1822,2013-04-13,Mumbai,329,0
1971,2013-08-31,Pune,105,0
2332,2014-02-16,Mumbai,206,0



Races with multiple winners:


,meet_date_parsed,venue,race_no,winner_count
594,2011-04-17,Mumbai,344,2
1791,2013-03-31,Mumbai,298,2
1915,2013-08-04,Pune,49,2
2054,2013-09-29,Pune,188,2
3007,2015-03-22,Mumbai,281,2
3066,2015-04-19,Mumbai,340,2
3233,2015-09-20,Pune,146,2
3282,2015-10-11,Pune,195,2
3473,2016-01-24,Mumbai,153,2
4634,2018-04-14,Mumbai,282,2


In [14]:
print("Unique raw odds values:", runners["odds"].nunique())

display(
    runners["odds"]
    .astype(str)
    .value_counts(dropna=False)
    .head(100)
)

Unique raw odds values: 363


odds
20-Jan    9501
15-Jan    5328
--        4538
12-Jan    4109
10-Jan    3547
25-Jan    2447
08-Jan    2205
07-Jan    1777
20        1757
30-Jan    1705
40/1      1542
15        1300
06-Jan    1298
09-Jan    1044
05-Jan    1027
50/1       902
11-Feb     806
09-Feb     805
12         763
10         735
13-Feb     704
04-Jan     699
07-Feb     634
03-Jan     553
05-Feb     544
13-Apr     505
09-Apr     491
02-Jan     466
25         464
11-Apr     457
60/1       436
11-Jan     428
17-Apr     417
15-Apr     395
13-Jan     393
15-Feb     392
8          388
16-Jan     384
18-Oct     352
14-Jan     323
7          320
17-Feb     311
9          277
35/1       262
13-Oct     258
03-Feb     250
16-Oct     250
11-Oct     244
14-Oct     232
6          223
17-Jan     222
5          221
19-Apr     215
01-Jan     210
21-Apr     206
17-Oct     197
5.5        176
12-Oct     174
4.5        170
19-Oct     169
25-Apr     164
6.5        164
4          152
26-Oct     135
18-Jan     133
30         133
3.5  

In [15]:
import re
import numpy as np
import pandas as pd

month_to_num = {
    "Jan": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "Jun": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12,
}

def parse_odds(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    # Missing
    if s in {"", "--", "-"}:
        return np.nan

    # Excel-corrupted fractional odds:
    # 20-Jan -> 20/1
    # 11-Feb -> 11/2
    # 18-Oct -> 18/10
    m = re.fullmatch(r"(\d+(?:\.\d+)?)-([A-Za-z]{3})", s)

    if m:
        numerator = float(m.group(1))
        denominator = month_to_num.get(m.group(2).title())

        if denominator is None:
            return np.nan

        return 1 + numerator / denominator

    # Normal fractional odds:
    # 40/1, 90/100, 36/10
    m = re.fullmatch(
        r"(\d+(?:\.\d+)?)/(\d+(?:\.\d+)?)",
        s
    )

    if m:
        numerator = float(m.group(1))
        denominator = float(m.group(2))

        if denominator == 0:
            return np.nan

        return 1 + numerator / denominator

    # Already decimal
    try:
        value = float(s)

        if value <= 0:
            return np.nan

        return 1 + value

    except ValueError:
        return np.nan

In [16]:
runners["odds_decimal"] = runners["odds"].apply(parse_odds)

print("Total runners:", len(runners))
print("Valid odds:", runners["odds_decimal"].notna().sum())
print("Missing/invalid odds:", runners["odds_decimal"].isna().sum())
print(
    "Odds coverage:",
    round(runners["odds_decimal"].notna().mean() * 100, 2),
    "%"
)

Total runners: 68201
Valid odds: 63654
Missing/invalid odds: 4547
Odds coverage: 93.33 %


In [17]:
# Create a race identifier
runners["race_id"] = (
    runners["meet_date"].astype(str) + "_" +
    runners["venue"].astype(str) + "_" +
    runners["race_no"].astype(str)
)

race_stats = runners.groupby("race_id").agg(
    runners=("horse_name", "size"),
    valid_odds=("odds_decimal", "count"),
    missing_odds=("odds_decimal", lambda x: x.isna().sum())
)

race_stats["odds_complete"] = race_stats["missing_odds"] == 0

print("Total races:", len(race_stats))
print("Races with complete odds:", race_stats["odds_complete"].sum())
print("Races with missing odds:", (~race_stats["odds_complete"]).sum())
print(
    "Complete race coverage:",
    round(race_stats["odds_complete"].mean() * 100, 2),
    "%"
)

display(race_stats["missing_odds"].value_counts().sort_index())

Total races: 7135
Races with complete odds: 6005
Races with missing odds: 1130
Complete race coverage: 84.16 %


missing_odds
0     6005
1      694
2       55
3        8
4       13
5       22
6       31
7       25
8       38
9       29
10      30
11      51
12      41
13      34
14      55
15       1
16       2
19       1
Name: count, dtype: int64

In [18]:
# Inspect races with missing odds
tmp = runners.copy()

# Try to interpret placing as numeric
tmp["placing_num"] = pd.to_numeric(tmp["placing"], errors="coerce")

race_check = tmp.groupby("race_id").agg(
    total_runners=("horse_name", "size"),
    missing_odds=("odds_decimal", lambda x: x.isna().sum()),
    numeric_placings=("placing_num", "count"),
    non_numeric_placings=("placing_num", lambda x: x.isna().sum()),
    unique_placings=("placing_num", lambda x: x.dropna().nunique()),
)

# Display races where odds are missing AND there are non-numeric placings
problem_races = race_check[
    (race_check["missing_odds"] > 0) &
    (race_check["non_numeric_placings"] > 0)
]

print("Races with missing odds:", (race_check["missing_odds"] > 0).sum())
print("Races with non-numeric placings:", (race_check["non_numeric_placings"] > 0).sum())
print(
    "Races with BOTH:",
    len(problem_races)
)

display(problem_races.head(30))

Races with missing odds: 1130
Races with non-numeric placings: 1596
Races with BOTH: 829


,total_runners,missing_odds,numeric_placings,non_numeric_placings,unique_placings
race_id,,,,,
01-01-2012_Mumbai_114,13,1,12,1,12
01-01-2017_Mumbai_88,7,1,6,1,6
01-01-2017_Mumbai_92,11,1,8,3,8
01-02-2020_Mumbai_116,14,1,12,2,12
01-02-2025_Mumbai_91,11,3,8,3,8
01-02-2025_Mumbai_92,11,1,10,1,10
01-02-2026_Mumbai_90,11,1,10,1,10
01-04-2012_Mumbai_312,9,1,8,1,8
01-04-2018_Mumbai_263,13,1,12,1,12


In [19]:
# What are the non-numeric placing values?
non_numeric = runners[
    pd.to_numeric(runners["placing"], errors="coerce").isna()
]["placing"]

print("Non-numeric placing values:")
display(
    non_numeric.astype(str)
    .value_counts(dropna=False)
)

Non-numeric placing values:


placing
WD      1115
DNC      372
WDRN     253
DNF       83
NDS       36
-         32
NPR       21
DQ        18
NS         9
Name: count, dtype: int64

In [20]:
# Patch the clearly corrupted fractional odds
def fix_zero_denominator(x):
    if pd.isna(x):
        return x

    s = str(x).strip()

    # 20/0 -> 20/1, 40/0 -> 40/1, etc.
    m = re.fullmatch(r"(\d+(?:\.\d+)?)/0", s)

    if m:
        return f"{m.group(1)}/1"

    return x

runners["odds_fixed"] = runners["odds"].apply(fix_zero_denominator)

# Re-parse using our existing parser
runners["odds_decimal"] = runners["odds_fixed"].apply(parse_odds)

tmp = runners.copy()
tmp["placing_num"] = pd.to_numeric(tmp["placing"], errors="coerce")

finishers = tmp["placing_num"].notna()

print("Actual finishers:", finishers.sum())
print("Missing odds among finishers:", tmp.loc[finishers, "odds_decimal"].isna().sum())
print(
    "Finisher odds coverage:",
    round(tmp.loc[finishers, "odds_decimal"].notna().mean() * 100, 2),
    "%"
)

Actual finishers: 66262
Missing odds among finishers: 3622
Finisher odds coverage: 94.53 %


In [21]:
# Race-level completeness among actual finishers only
tmp = runners.copy()
tmp["placing_num"] = pd.to_numeric(tmp["placing"], errors="coerce")

finishers = tmp[tmp["placing_num"].notna()].copy()

race_odds = finishers.groupby("race_id").agg(
    finishers=("horse_name", "size"),
    valid_odds=("odds_decimal", "count"),
    missing_odds=("odds_decimal", lambda x: x.isna().sum())
)

race_odds["complete"] = race_odds["missing_odds"] == 0

print("Total races:", len(race_odds))
print("Races with complete finisher odds:", race_odds["complete"].sum())
print("Races with missing finisher odds:", (~race_odds["complete"]).sum())
print(
    "Race-level odds coverage:",
    round(race_odds["complete"].mean() * 100, 2),
    "%"
)

display(race_odds["missing_odds"].value_counts().sort_index())

Total races: 7131
Races with complete finisher odds: 6736
Races with missing finisher odds: 395
Race-level odds coverage: 94.46 %


missing_odds
0     6736
1       20
3        3
4       13
5       27
6       32
7       32
8       34
9       28
10      40
11      48
12      41
13      39
14      34
15       2
16       1
17       1
Name: count, dtype: int64

In [22]:
complete_races = race_odds.index[race_odds["complete"]]

clean = runners[
    runners["race_id"].isin(complete_races)
].copy()

clean["placing_num"] = pd.to_numeric(clean["placing"], errors="coerce")

# Only actual finishers
clean = clean[clean["placing_num"].notna()]

winner_check = clean.groupby("race_id").agg(
    runners=("horse_name", "size"),
    winners=("placing_num", lambda x: (x == 1).sum()),
    unique_placings=("placing_num", "nunique"),
    max_placing=("placing_num", "max")
)

print("Complete races:", len(winner_check))
print("Exactly one winner:", (winner_check["winners"] == 1).sum())
print("Races with != 1 winner:", (winner_check["winners"] != 1).sum())
print("Races with duplicate placings:", (winner_check["unique_placings"] != winner_check["runners"]).sum())

display(winner_check[winner_check["winners"] != 1].head(20))

Complete races: 6736
Exactly one winner: 6723
Races with != 1 winner: 13
Races with duplicate placings: 57


,runners,winners,unique_placings,max_placing
race_id,,,,
04-08-2013_Pune_49,9,2,8,9.0
11-10-2015_Pune_195,8,2,7,8.0
13-04-2019_Mumbai_238,9,2,8,9.0
14-04-2018_Mumbai_282,8,2,7,8.0
16-01-2020_Mumbai_94,12,2,11,12.0
16-12-2018_Mumbai_54,11,2,10,11.0
17-04-2011_Mumbai_344,5,2,4,5.0
19-04-2015_Mumbai_340,9,2,8,9.0
20-09-2015_Pune_146,7,2,6,7.0


In [23]:
# Keep only the data needed for the research/analysis notebook

analysis = runners[
    runners["race_id"].isin(complete_races)
].copy()

analysis["placing_num"] = pd.to_numeric(
    analysis["placing"],
    errors="coerce"
)

# Only actual finishers
analysis = analysis[analysis["placing_num"].notna()].copy()

# Columns needed going forward
analysis = analysis[
    [
        "race_id",
        "meet_date",
        "venue",
        "race_no",
        "horse_name",
        "horse_seq",
        "placing_num",
        "odds_decimal",
    ]
]

analysis.to_csv(
    "../../data/cleaned/mvp/cleaned_data_v1.csv",
    index=False
)

print("Saved:", len(analysis), "rows")
print("Races:", analysis["race_id"].nunique())
print("Columns:", list(analysis.columns))

Saved: 62559 rows
Races: 6736
Columns: ['race_id', 'meet_date', 'venue', 'race_no', 'horse_name', 'horse_seq', 'placing_num', 'odds_decimal']
